In [11]:
#####Bad Pixels not masked###

import os
import time
import arcpy
from arcpy.sa import ExtractByMask

# -------------------------------------------------
# USER SETTINGS
# -------------------------------------------------
in_folder    = r"E:\Mosaic_Norge\Test"  # mosaics to be masked
mask_raster  = r"E:\Mosaic_Norge\snap_raster\dtm_mask_srast_fixed.tif"

# ✅ requested output folder
out_folder   = r"E:\Mosaic_Norge\Final_rasts"
out_suffix   = "_masked"

TARGET_CELLSIZE = 10  # meters

build_stats     = True
build_pyramids  = True
overwrite       = True
# -------------------------------------------------

arcpy.env.overwriteOutput = overwrite
os.makedirs(out_folder, exist_ok=True)

def log(msg):
    print(msg)
    try:
        arcpy.AddMessage(msg)
    except:
        pass

def format_eta(seconds):
    seconds = max(0, int(seconds))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    if h > 0:
        return f"{h}h {m}m {s}s"
    if m > 0:
        return f"{m}m {s}s"
    return f"{s}s"

def progress_bar(i, n, width=28):
    if n <= 0:
        return ""
    frac = i / n
    filled = int(round(frac * width))
    bar = "█" * filled + "░" * (width - filled)
    return f"[{bar}] {i}/{n} ({int(frac*100)}%)"

def signature(r):
    d = arcpy.Describe(r)
    ext = d.extent
    return (
        d.spatialReference.factoryCode if d.spatialReference else None,
        round(d.meanCellWidth, 6),
        round(d.meanCellHeight, 6),
        round(ext.XMin, 6),
        round(ext.YMax, 6),
        int(d.width),
        int(d.height)
    )

# -------------------------------------------------
# Checks + environment lock
# -------------------------------------------------
if not arcpy.Exists(mask_raster):
    raise RuntimeError(f"Mask raster not found: {mask_raster}")

try:
    arcpy.CheckOutExtension("Spatial")
except:
    raise RuntimeError("Spatial Analyst extension is not available. ExtractByMask requires it.")

mask_sig = signature(mask_raster)
mask_desc = arcpy.Describe(mask_raster)
mask_ext  = mask_desc.extent
mask_sr   = mask_desc.spatialReference
mask_cell = mask_desc.meanCellWidth

if round(mask_cell, 6) != float(TARGET_CELLSIZE):
    log(f"WARNING: mask cellsize is {mask_cell}, but you requested {TARGET_CELLSIZE}. "
        f"Outputs will be forced to {TARGET_CELLSIZE}m.")

# 🔒 Lock to mask grid
arcpy.env.snapRaster = mask_raster
arcpy.env.mask = mask_raster
arcpy.env.extent = mask_ext
arcpy.env.cellSize = TARGET_CELLSIZE
arcpy.env.outputCoordinateSystem = mask_sr

# Collect inputs
tifs = [f for f in os.listdir(in_folder) if f.lower().endswith((".tif", ".tiff"))]
if not tifs:
    raise RuntimeError(f"No .tif/.tiff files found in: {in_folder}")

tifs = sorted(tifs)

log(f"Found {len(tifs)} rasters in: {in_folder}")
log(f"Mask/snap raster: {mask_raster}")
log(f"Output folder   : {out_folder}")
log(f"Forced cell size: {TARGET_CELLSIZE} m")
log(f"Mask signature  : {mask_sig}")

# -------------------------------------------------
# Run
# -------------------------------------------------
t0_global = time.time()
times = []

failed = []
mismatch = []

for idx, fn in enumerate(tifs, start=1):
    in_r = os.path.join(in_folder, fn)
    base, _ = os.path.splitext(fn)
    out_r = os.path.join(out_folder, f"{base}{out_suffix}.tif")

    # If overwrite is False and output exists
    if arcpy.Exists(out_r) and not overwrite:
        dt = 0.0
        # still verify alignment if it exists
        try:
            out_sig = signature(out_r)
            if out_sig != mask_sig:
                mismatch.append((out_r, out_sig))
        except:
            mismatch.append((out_r, "could not read signature"))
        avg = (sum(times) / len(times)) if times else 0
        eta = format_eta((len(tifs) - idx) * avg) if avg > 0 else "?"
        log(f"{progress_bar(idx, len(tifs))} EXISTS (skip) | ETA: {eta} | {fn}")
        continue

    t1 = time.time()
    try:
        masked = ExtractByMask(in_r, mask_raster)
        masked.save(out_r)

        if build_stats:
            arcpy.management.CalculateStatistics(out_r, 1, 1, "OVERWRITE")
        if build_pyramids:
            arcpy.management.BuildPyramids(out_r)

        # ✅ Check alignment/dimensions match mask
        out_sig = signature(out_r)
        if out_sig != mask_sig:
            mismatch.append((out_r, out_sig))

    except Exception as e:
        failed.append((in_r, str(e)))
        log(f"{progress_bar(idx, len(tifs))} FAILED | {fn}\n    {e}")
        continue

    dt = time.time() - t1
    times.append(dt)
    avg = sum(times) / len(times)
    eta = format_eta((len(tifs) - idx) * avg)

    # brief per-file log
    ok_txt = "OK" if (out_sig == mask_sig) else "ALIGNMENT MISMATCH"
    log(f"{progress_bar(idx, len(tifs))} {ok_txt} | {dt:.1f}s | ETA: {eta} | {fn}")

# -------------------------------------------------
# Summary
# -------------------------------------------------
total = time.time() - t0_global
log("\n" + "=" * 70)
log("DONE")
log(f"Total runtime: {total/60:.2f} minutes")
log(f"Outputs in   : {out_folder}")
log(f"Mask signature: {mask_sig}")

if mismatch:
    log("\nWARNING: Some outputs do NOT exactly match the mask grid signature (SR, cell, XMin, YMax, width, height):")
    for p, sig in mismatch[:20]:
        log(f" - {p} -> {sig}")
    if len(mismatch) > 20:
        log(f" ... and {len(mismatch)-20} more")
else:
    log("\nAll masked mosaics match the mask alignment + dimensions exactly ✅")

if failed:
    log("\nFAILED files:")
    for p, err in failed[:20]:
        log(f" - {p}\n   {err}")
    if len(failed) > 20:
        log(f" ... and {len(failed)-20} more")
log("=" * 70)


Found 1 rasters in: E:\Mosaic_Norge\Test
Mask/snap raster: E:\Mosaic_Norge\snap_raster\dtm_mask_srast_fixed.tif
Output folder   : E:\Mosaic_Norge\Final_rasts
Forced cell size: 10 m
Mask signature  : (32633, 10.0, 10.0, -79745.000009, 7942295.00005, 119486, 149587)
[████████████████████████████] 1/1 (100%) OK | 2947.4s | ETA: 0s | tri_mosaic.tif

DONE
Total runtime: 49.12 minutes
Outputs in   : E:\Mosaic_Norge\Final_rasts
Mask signature: (32633, 10.0, 10.0, -79745.000009, 7942295.00005, 119486, 149587)

All masked mosaics match the mask alignment + dimensions exactly ✅


In [1]:
#####Bad Pixels masked####

import os
import time
import arcpy
from arcpy.sa import ExtractByMask, SetNull, Raster  # <-- ADDED SetNull + Raster

# -------------------------------------------------
# USER SETTINGS
# -------------------------------------------------
in_folder    = r"E:\Mosaic_Norge\Test"  # mosaics to be masked
mask_raster  = r"E:\Mosaic_Norge\snap_raster\dtm_mask_srast_fixed.tif"

# ✅ requested output folder
out_folder   = r"E:\Mosaic_Norge\Final_rasts"
out_suffix   = "_masked"

TARGET_CELLSIZE = 10  # meters

build_stats     = True
build_pyramids  = True
overwrite       = True

# --- ADDED: threshold for "bad" pixels ---
BAD_THRESHOLD = 1e15
# -------------------------------------------------

arcpy.env.overwriteOutput = overwrite
os.makedirs(out_folder, exist_ok=True)

def log(msg):
    print(msg)
    try:
        arcpy.AddMessage(msg)
    except:
        pass

def format_eta(seconds):
    seconds = max(0, int(seconds))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    if h > 0:
        return f"{h}h {m}m {s}s"
    if m > 0:
        return f"{m}m {s}s"
    return f"{s}s"

def progress_bar(i, n, width=28):
    if n <= 0:
        return ""
    frac = i / n
    filled = int(round(frac * width))
    bar = "█" * filled + "░" * (width - filled)
    return f"[{bar}] {i}/{n} ({int(frac*100)}%)"

def signature(r):
    d = arcpy.Describe(r)
    ext = d.extent
    return (
        d.spatialReference.factoryCode if d.spatialReference else None,
        round(d.meanCellWidth, 6),
        round(d.meanCellHeight, 6),
        round(ext.XMin, 6),
        round(ext.YMax, 6),
        int(d.width),
        int(d.height)
    )

# -------------------------------------------------
# Checks + environment lock
# -------------------------------------------------
if not arcpy.Exists(mask_raster):
    raise RuntimeError(f"Mask raster not found: {mask_raster}")

try:
    arcpy.CheckOutExtension("Spatial")
except:
    raise RuntimeError("Spatial Analyst extension is not available. ExtractByMask requires it.")

mask_sig = signature(mask_raster)
mask_desc = arcpy.Describe(mask_raster)
mask_ext  = mask_desc.extent
mask_sr   = mask_desc.spatialReference
mask_cell = mask_desc.meanCellWidth

if round(mask_cell, 6) != float(TARGET_CELLSIZE):
    log(f"WARNING: mask cellsize is {mask_cell}, but you requested {TARGET_CELLSIZE}. "
        f"Outputs will be forced to {TARGET_CELLSIZE}m.")

# 🔒 Lock to mask grid
arcpy.env.snapRaster = mask_raster
arcpy.env.mask = mask_raster
arcpy.env.extent = mask_ext
arcpy.env.cellSize = TARGET_CELLSIZE
arcpy.env.outputCoordinateSystem = mask_sr

# Collect inputs
tifs = [f for f in os.listdir(in_folder) if f.lower().endswith((".tif", ".tiff"))]
if not tifs:
    raise RuntimeError(f"No .tif/.tiff files found in: {in_folder}")

tifs = sorted(tifs)

log(f"Found {len(tifs)} rasters in: {in_folder}")
log(f"Mask/snap raster: {mask_raster}")
log(f"Output folder   : {out_folder}")
log(f"Forced cell size: {TARGET_CELLSIZE} m")
log(f"Mask signature  : {mask_sig}")
log(f"Bad-pixel rule  : values > {BAD_THRESHOLD} -> NoData")

# -------------------------------------------------
# Run
# -------------------------------------------------
t0_global = time.time()
times = []

failed = []
mismatch = []

for idx, fn in enumerate(tifs, start=1):
    in_r = os.path.join(in_folder, fn)
    base, _ = os.path.splitext(fn)
    out_r = os.path.join(out_folder, f"{base}{out_suffix}.tif")

    # If overwrite is False and output exists
    if arcpy.Exists(out_r) and not overwrite:
        dt = 0.0
        # still verify alignment if it exists
        try:
            out_sig = signature(out_r)
            if out_sig != mask_sig:
                mismatch.append((out_r, out_sig))
        except:
            mismatch.append((out_r, "could not read signature"))
        avg = (sum(times) / len(times)) if times else 0
        eta = format_eta((len(tifs) - idx) * avg) if avg > 0 else "?"
        log(f"{progress_bar(idx, len(tifs))} EXISTS (skip) | ETA: {eta} | {fn}")
        continue

    t1 = time.time()
    try:
        masked = ExtractByMask(in_r, mask_raster)

        # --- ADDED: kill inland artifacts / bogus huge values ---
        # Any cell > BAD_THRESHOLD becomes NoData
        masked = SetNull(Raster(masked) > BAD_THRESHOLD, Raster(masked))

        masked.save(out_r)

        if build_stats:
            arcpy.management.CalculateStatistics(out_r, 1, 1, "OVERWRITE")
        if build_pyramids:
            arcpy.management.BuildPyramids(out_r)

        # ✅ Check alignment/dimensions match mask
        out_sig = signature(out_r)
        if out_sig != mask_sig:
            mismatch.append((out_r, out_sig))

    except Exception as e:
        failed.append((in_r, str(e)))
        log(f"{progress_bar(idx, len(tifs))} FAILED | {fn}\n    {e}")
        continue

    dt = time.time() - t1
    times.append(dt)
    avg = sum(times) / len(times)
    eta = format_eta((len(tifs) - idx) * avg)

    # brief per-file log
    ok_txt = "OK" if (out_sig == mask_sig) else "ALIGNMENT MISMATCH"
    log(f"{progress_bar(idx, len(tifs))} {ok_txt} | {dt:.1f}s | ETA: {eta} | {fn}")

# -------------------------------------------------
# Summary
# -------------------------------------------------
total = time.time() - t0_global
log("\n" + "=" * 70)
log("DONE")
log(f"Total runtime: {total/60:.2f} minutes")
log(f"Outputs in   : {out_folder}")
log(f"Mask signature: {mask_sig}")

if mismatch:
    log("\nWARNING: Some outputs do NOT exactly match the mask grid signature (SR, cell, XMin, YMax, width, height):")
    for p, sig in mismatch[:20]:
        log(f" - {p} -> {sig}")
    if len(mismatch) > 20:
        log(f" ... and {len(mismatch)-20} more")
else:
    log("\nAll masked mosaics match the mask alignment + dimensions exactly ✅")

if failed:
    log("\nFAILED files:")
    for p, err in failed[:20]:
        log(f" - {p}\n   {err}")
    if len(failed) > 20:
        log(f" ... and {len(failed)-20} more")
log("=" * 70)


Found 1 rasters in: E:\Mosaic_Norge\Test
Mask/snap raster: E:\Mosaic_Norge\snap_raster\dtm_mask_srast_fixed.tif
Output folder   : E:\Mosaic_Norge\Final_rasts
Forced cell size: 10 m
Mask signature  : (32633, 10.0, 10.0, -79745.000009, 7942295.00005, 119486, 149587)
Bad-pixel rule  : values > 1000000000000000.0 -> NoData
[████████████████████████████] 1/1 (100%) FAILED | tri_mosaic.tif
    ERROR 010240: Could not save raster dataset to tri_mosaic_masked.tif with output format TIFF.

DONE
Total runtime: 19.13 minutes
Outputs in   : E:\Mosaic_Norge\Final_rasts
Mask signature: (32633, 10.0, 10.0, -79745.000009, 7942295.00005, 119486, 149587)

All masked mosaics match the mask alignment + dimensions exactly ✅

FAILED files:
 - E:\Mosaic_Norge\Test\tri_mosaic.tif
   ERROR 010240: Could not save raster dataset to tri_mosaic_masked.tif with output format TIFF.


In [1]:
#####Eliminating bad pixels in Land + labelling rasters that have bad pixels as _ch#####


###NOtes check threshold of:
#tri has very high values as upper end 574.454. Adjust
# Spi, threshold has to be update to <1e9 
#All the curvatrures are in the thousands for upper and lower values. the thresholds 
# have to be adjusted to perhaps 100's inspect the rasters
##Aspec has -1 values those have to be converrted to -1

import os
import time
import arcpy
from arcpy.sa import ExtractByMask, SetNull, Raster

# -------------------------------------------------
# USER SETTINGS
# -------------------------------------------------
in_folder    = r"F:\Topodata_basins\Mosaic_Norge\Test"
mask_raster  = r"F:\Topodata_basins\Mosaic_Norge\snap_raster\dtm_mask_srast_fixed.tif"

out_folder   = r"F:\Topodata_basins\Mosaic_Norge\Final_rasts"
out_suffix   = "_masked"

TARGET_CELLSIZE = 10

build_stats     = True
build_pyramids  = True
overwrite       = True

BAD_THRESHOLD = 100
# -------------------------------------------------

arcpy.env.overwriteOutput = overwrite
os.makedirs(out_folder, exist_ok=True)

def log(msg):
    print(msg)
    try:
        arcpy.AddMessage(msg)
    except:
        pass

def format_eta(seconds):
    seconds = max(0, int(seconds))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    if h > 0:
        return f"{h}h {m}m {s}s"
    if m > 0:
        return f"{m}m {s}s"
    return f"{s}s"

def progress_bar(i, n, width=28):
    frac = i / n
    filled = int(round(frac * width))
    bar = "█" * filled + "░" * (width - filled)
    return f"[{bar}] {i}/{n} ({int(frac*100)}%)"

def signature(r):
    d = arcpy.Describe(r)
    ext = d.extent
    return (
        d.spatialReference.factoryCode if d.spatialReference else None,
        round(d.meanCellWidth, 6),
        round(d.meanCellHeight, 6),
        round(ext.XMin, 6),
        round(ext.YMax, 6),
        int(d.width),
        int(d.height)
    )

# -------------------------------------------------
# Environment lock
# -------------------------------------------------
arcpy.env.snapRaster = mask_raster
arcpy.env.mask = mask_raster
arcpy.env.extent = arcpy.Describe(mask_raster).extent
arcpy.env.cellSize = TARGET_CELLSIZE
arcpy.env.outputCoordinateSystem = arcpy.Describe(mask_raster).spatialReference

# Collect inputs
tifs = sorted(f for f in os.listdir(in_folder) if f.lower().endswith(".tif"))

log(f"Found {len(tifs)} rasters")
log(f"Bad-pixel rule: values > {BAD_THRESHOLD} → NoData + '_ch' suffix")

# -------------------------------------------------
# Run
# -------------------------------------------------
t0_global = time.time()
times = []

for idx, fn in enumerate(tifs, start=1):
    in_r = os.path.join(in_folder, fn)
    base, _ = os.path.splitext(fn)

    t1 = time.time()

    try:
        # ---------- NEW: detect bad pixels ----------
        bad_test = Raster(in_r) > BAD_THRESHOLD
        has_bad = float(arcpy.management.GetRasterProperties(
            bad_test, "MAXIMUM"
        )[0]) > 0
        # -------------------------------------------

        # build output name
        suffix = out_suffix + ("_ch" if has_bad else "")
        out_r = os.path.join(out_folder, f"{base}{suffix}.tif")

        masked = ExtractByMask(in_r, mask_raster)

        if has_bad:
            masked = SetNull(Raster(masked) > BAD_THRESHOLD, Raster(masked))

        masked.save(out_r)

        if build_stats:
            arcpy.management.CalculateStatistics(out_r, 1, 1, "OVERWRITE")
        if build_pyramids:
            arcpy.management.BuildPyramids(out_r)

        out_sig = signature(out_r)

    except Exception as e:
        log(f"{progress_bar(idx, len(tifs))} FAILED | {fn}\n    {e}")
        continue

    dt = time.time() - t1
    times.append(dt)
    avg = sum(times) / len(times)
    eta = format_eta((len(tifs) - idx) * avg)

    tag = "CH" if has_bad else "OK"
    log(f"{progress_bar(idx, len(tifs))} {tag} | {dt:.1f}s | ETA: {eta} | {fn}")

# -------------------------------------------------
# Summary
# -------------------------------------------------
total = time.time() - t0_global
log("\n" + "=" * 70)
log("DONE")
log(f"Total runtime: {total/60:.2f} minutes")
log(f"Outputs in   : {out_folder}")
log("=" * 70)


Found 2 rasters
Bad-pixel rule: values > 100 → NoData + '_ch' suffix
[██████████████░░░░░░░░░░░░░░] 1/2 (50%) CH | 3936.1s | ETA: 1h 5m 36s | spi_mosaic.tif
[████████████████████████████] 2/2 (100%) CH | 3947.5s | ETA: 0s | tri_mosaic.tif

DONE
Total runtime: 131.39 minutes
Outputs in   : F:\Topodata_basins\Mosaic_Norge\Final_rasts


In [1]:
#####Eliminating bad pixels in Land + labelling rasters that have bad pixels as _ch + THRESHOLD EDITED#####

import os
import time
import arcpy
from arcpy.sa import ExtractByMask, SetNull, Raster

# -------------------------------------------------
# USER SETTINGS
# -------------------------------------------------
in_folder    = r"F:\Topodata_basins\Mosaic_Norge\Test"
mask_raster  = r"F:\Topodata_basins\Mosaic_Norge\snap_raster\dtm_mask_srast_fixed.tif"

out_folder   = r"F:\Topodata_basins\Mosaic_Norge\Final_rasts\Ready"
out_suffix   = "_masked"

TARGET_CELLSIZE = 10

build_stats     = True
build_pyramids  = True
overwrite       = True

# -------------------------------------------------
# Threshold rules (substring match in tif name)
# -------------------------------------------------
THRESH_RULES = {
    # tri & spi: values > 100 are bad
    "tri": {"type": "upper", "upper": 100},
    "spi": {"type": "upper", "upper": 100},

    # slope & dtm: values > 1e9 are bad
    "slope": {"type": "upper", "upper": 1e9},
    "dtm":   {"type": "upper", "upper": 1e9},

    # curvatures: values outside [-15, 15] are bad
    "curv_plan": {"type": "range", "lower": -15, "upper": 15},
    "curv_prof": {"type": "range", "lower": -15, "upper": 15},
    "curv_tot":  {"type": "range", "lower": -15, "upper": 15},

    # aspect: values < 0 OR values > 1e9 are bad
    "aspect": {"type": "range", "lower": 0, "upper": 1e9},
}

# Fallback if no rule matched (safe default)
DEFAULT_RULE = {"type": "upper", "upper": 1e9}
# -------------------------------------------------

arcpy.env.overwriteOutput = overwrite
os.makedirs(out_folder, exist_ok=True)

def log(msg):
    print(msg)
    try:
        arcpy.AddMessage(msg)
    except:
        pass

def format_eta(seconds):
    seconds = max(0, int(seconds))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    if h > 0:
        return f"{h}h {m}m {s}s"
    if m > 0:
        return f"{m}m {s}s"
    return f"{s}s"

def progress_bar(i, n, width=28):
    frac = i / n
    filled = int(round(frac * width))
    bar = "█" * filled + "░" * (width - filled)
    return f"[{bar}] {i}/{n} ({int(frac*100)}%)"

def signature(r):
    d = arcpy.Describe(r)
    ext = d.extent
    return (
        d.spatialReference.factoryCode if d.spatialReference else None,
        round(d.meanCellWidth, 6),
        round(d.meanCellHeight, 6),
        round(ext.XMin, 6),
        round(ext.YMax, 6),
        int(d.width),
        int(d.height)
    )

def get_rule_for_filename(filename):
    """Pick the first matching rule by substring; otherwise DEFAULT_RULE."""
    name = filename.lower()
    for key, rule in THRESH_RULES.items():
        if key in name:
            return key, rule
    return None, DEFAULT_RULE

def bad_mask(raster_obj, rule):
    """
    Returns a boolean raster where True indicates 'bad' pixels, based on rule.
    - upper: bad if value > upper
    - lower: bad if value < lower
    - range: bad if value < lower OR value > upper
    """
    rtype = rule.get("type")
    if rtype == "upper":
        return raster_obj > rule["upper"]
    elif rtype == "lower":
        return raster_obj < rule["lower"]
    elif rtype == "range":
        return (raster_obj < rule["lower"]) | (raster_obj > rule["upper"])
    else:
        raise ValueError(f"Unknown rule type: {rtype}")

# -------------------------------------------------
# Environment lock
# -------------------------------------------------
arcpy.env.snapRaster = mask_raster
arcpy.env.mask = mask_raster
arcpy.env.extent = arcpy.Describe(mask_raster).extent
arcpy.env.cellSize = TARGET_CELLSIZE
arcpy.env.outputCoordinateSystem = arcpy.Describe(mask_raster).spatialReference

# Collect inputs
tifs = sorted(f for f in os.listdir(in_folder) if f.lower().endswith(".tif"))

log(f"Found {len(tifs)} rasters")
log("Bad-pixel rules (by filename substring):")
log("  tri/spi      : value > 100               -> NoData")
log("  slope/dtm    : value > 1e9               -> NoData")
log("  curv_*       : value outside [-15, 15]   -> NoData")
log("  aspect       : value < 0 or value > 1e9  -> NoData")
log("  default      : value > 1e9               -> NoData (if no name match)")

# -------------------------------------------------
# Run
# -------------------------------------------------
t0_global = time.time()
times = []

for idx, fn in enumerate(tifs, start=1):
    in_r = os.path.join(in_folder, fn)
    base, _ = os.path.splitext(fn)

    t1 = time.time()

    try:
        rule_key, rule = get_rule_for_filename(fn)

        r_in = Raster(in_r)

        # ---------- detect bad pixels (rule-specific) ----------
        bad_test = bad_mask(r_in, rule)
        has_bad = float(arcpy.management.GetRasterProperties(
            bad_test, "MAXIMUM"
        )[0]) > 0
        # ------------------------------------------------------

        # build output name
        suffix = out_suffix + ("_ch" if has_bad else "")
        out_r = os.path.join(out_folder, f"{base}{suffix}.tif")

        masked = ExtractByMask(in_r, mask_raster)

        if has_bad:
            # ---------- remove bad pixels (rule-specific) ----------
            masked_r = Raster(masked)
            masked = SetNull(bad_mask(masked_r, rule), masked_r)
            # ------------------------------------------------------

        masked.save(out_r)

        if build_stats:
            arcpy.management.CalculateStatistics(out_r, 1, 1, "OVERWRITE")
        if build_pyramids:
            arcpy.management.BuildPyramids(out_r)

        out_sig = signature(out_r)

    except Exception as e:
        log(f"{progress_bar(idx, len(tifs))} FAILED | {fn}\n    {e}")
        continue

    dt = time.time() - t1
    times.append(dt)
    avg = sum(times) / len(times)
    eta = format_eta((len(tifs) - idx) * avg)

    tag = "CH" if has_bad else "OK"
    rule_tag = rule_key if rule_key else "default"
    log(f"{progress_bar(idx, len(tifs))} {tag} | {dt:.1f}s | ETA: {eta} | rule={rule_tag} | {fn}")

# -------------------------------------------------
# Summary
# -------------------------------------------------
total = time.time() - t0_global
log("\n" + "=" * 70)
log("DONE")
log(f"Total runtime: {total/60:.2f} minutes")
log(f"Outputs in   : {out_folder}")
log("=" * 70)



Found 3 rasters
Bad-pixel rules (by filename substring):
  tri/spi      : value > 100               -> NoData
  slope/dtm    : value > 1e9               -> NoData
  curv_*       : value outside [-15, 15]   -> NoData
  aspect       : value < 0 or value > 1e9  -> NoData
  default      : value > 1e9               -> NoData (if no name match)
[█████████░░░░░░░░░░░░░░░░░░░] 1/3 (33%) CH | 4363.8s | ETA: 2h 25m 27s | rule=aspect | aspect_mosaic.tif
[███████████████████░░░░░░░░░] 2/3 (66%) CH | 4348.9s | ETA: 1h 12m 36s | rule=curv_plan | curv_plan_mosaic.tif
[████████████████████████████] 3/3 (100%) CH | 4686.7s | ETA: 0s | rule=curv_prof | curv_prof_mosaic.tif

DONE
Total runtime: 223.32 minutes
Outputs in   : F:\Topodata_basins\Mosaic_Norge\Final_rasts\Ready
